# Demostracion Rapida e Interactiva: Pipeline SAR Drones (SAREnv + MTS)

Este cuaderno proporciona una demostracion agil, interactiva y de extremo a extremo para evaluar estrategias de busqueda y rescate (Search and Rescue - SAR) con drones sobre escenarios reales (Casa de Campo de Madrid):

1. **Paso 1: Setup y Contexto Geografico:** Poligono delimitador en coordenadas reales y vista aerea satelital anotada.
2. **Paso 2: Generador Interactivo de Probabilidad:** Sliders segun perfiles del manual de Robert Koester (Lost Person Behavior) con filtros de transitabilidad fisica (P=0.0 en lagos/estructuras).
3. **Paso 3: Seleccion, Ejecucion y Visualizacion Dinamica SAR:** Selector para lanzar cualquier algoritmo (SAREnv continuo o MTS bioinspirado), renderizado de la trayectoria con consumo del mapa de creencias b(v^k) y cuadro de evaluacion SAR en directo.

> **Arquitectura Modular del Framework (`sarenv-mts`):**
> Las operaciones computacionales se importan directamente desde los paquetes centrales integrados en el framework:
> - `sarenv`: Paquete nativo de generacion cartografica y distribuciones bayesianas LPB.
> - `middleware`: Herramientas de union SAREnv <-> MTS, filtros fisicos OSM y transformaciones UTM.
> - `extra.visualizacion_pro`: Calculo dinamico de la huella del sensor (50 m) y mapas de creencias residuales b(v^k).
> - `metrics`: Evaluador unificado de rendimiento SAR (`PathEvaluatorTFM`).

In [ ]:
import os
import sys
import json
import glob
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from IPython import get_ipython
import shapely.geometry
from shapely.geometry import Point, LineString, Polygon

# Forzar backend interactivo inline en Jupyter
ipy = get_ipython()
if ipy is not None:
    ipy.run_line_magic('matplotlib', 'inline')

# Resolucion dinamica y automatica de rutas (detecta la raiz del framework MTS)
search_dir = os.path.abspath(os.getcwd())
mts_root = None
while search_dir and search_dir != os.path.dirname(search_dir):
    if os.path.isfile(os.path.join(search_dir, "bf-busqueda.py")) and os.path.isdir(os.path.join(search_dir, "sarenv")):
        mts_root = search_dir
        if search_dir not in sys.path:
            sys.path.insert(0, search_dir)
        break
    search_dir = os.path.dirname(search_dir)

if mts_root and os.getcwd() != mts_root:
    os.chdir(mts_root)

# Imports nativos de los modulos del framework profesional
import sarenv
import sarenv.analytics.paths as sap
import middleware.utils_pipeline as utils
import extra.visualizacion_pro as vis_pro
import extra.generar_figuras_memoria as fig_mem
from metrics import PathEvaluatorTFM

print(f"Framework sarenv-mts cargado correctamente. Directorio de trabajo: {os.getcwd()}")

---
## Paso 1: Setup y Contexto Geografico
Definicion explicita del poligono de busqueda en coordenadas geograficas WGS84 (Longitud, Latitud) y visualizacion del contexto aereo satelital con los elementos clave identificados (Lago, Parque de Atracciones, Zoo y Edificaciones).

In [ ]:
# 1. Definicion explicita del poligono de busqueda (Casa de Campo de Madrid)
CASA_DE_CAMPO_POLY = shapely.geometry.Polygon([
    [-3.753046089833776, 40.44343778644211],  # Vertice Norte (Aravaca)
    [-3.771067897299357, 40.43808936530237],  # Vertice Noroeste (Carretera de Castilla)
    [-3.780967823387276, 40.41887000168707],  # Vertice Suroeste (Somosaguas)
    [-3.773618938561301, 40.40203996553611],  # Vertice Sur (Aluche/Campamento)
    [-3.724145972222016, 40.41588490845162],  # Vertice Este (Madrid Rio / Lago)
    [-3.753046089833776, 40.44343778644211]   # Cierre del poligono
])

# 2. Visualizacion Satelite de Contexto Geografico con Anotaciones
print("Mostrando vista aerea satelital y elementos del entorno...")
fig, ax = fig_mem.plot_filtro_restricciones_satelite(perfil="autista")
display(fig)
plt.close(fig)

---
## Paso 2: Generador Interactivo de Probabilidad (Perfil Koester y Restricciones)
Ajuste en tiempo real de las ponderaciones de capas geograficas del manual de Robert Koester (Lost Person Behavior).
El generador calcula la distribucion bayesiana inicial con los radios de movilidad correspondientes y aplica automaticamente el filtro fisico de restricciones anulando a 0.0 la probabilidad en masas de agua y edificaciones.

In [ ]:
# Interfaz Interactiva de Seleccion y Ponderacion de Capas (Koester)

# Botones rapidos para cargar perfiles oficiales del manual
btn_autista = widgets.Button(description="Autista (5.12.8)", button_style="info")
btn_demencia = widgets.Button(description="Demencia (5.12.6)", button_style="warning")
btn_senderista = widgets.Button(description="Senderista (5.12.11)", button_style="success")

# Sliders para cada capa fisica
slider_structure = widgets.FloatSlider(value=0.45, min=0.0, max=1.0, step=0.01, description="Estructuras:")
slider_road = widgets.FloatSlider(value=0.18, min=0.0, max=1.0, step=0.01, description="Carreteras:")
slider_woodland = widgets.FloatSlider(value=0.09, min=0.0, max=1.0, step=0.01, description="Bosques:")
slider_water = widgets.FloatSlider(value=0.09, min=0.0, max=1.0, step=0.01, description="Agua:")
slider_field = widgets.FloatSlider(value=0.09, min=0.0, max=1.0, step=0.01, description="Campo:")
slider_brush = widgets.FloatSlider(value=0.09, min=0.0, max=1.0, step=0.01, description="Matorral:")

lbl_radios = widgets.Label(value="Radios cuartiles [25%, 50%, 75%, 95%]: [0.6, 1.6, 3.7, 15.2] km")
btn_generar_mapa = widgets.Button(description="Generar Mapa con Restricciones", button_style="primary")
out_paso2 = widgets.Output()

# Estado global de radios y perfil seleccionado
config_perfil = {
    "radios": [0.6, 1.6, 3.7, 15.2],
    "nombre": "autista"
}

def set_perfil_autista(b):
    slider_structure.value = 0.45
    slider_road.value = 0.18
    slider_water.value = 0.09
    slider_brush.value = 0.09
    slider_woodland.value = 0.09
    slider_field.value = 0.09
    config_perfil["radios"] = [0.6, 1.6, 3.7, 15.2]
    config_perfil["nombre"] = "autista"
    lbl_radios.value = "Radios cuartiles [25%, 50%, 75%, 95%]: [0.6, 1.6, 3.7, 15.2] km"

def set_perfil_demencia(b):
    slider_structure.value = 0.20
    slider_road.value = 0.18
    slider_woodland.value = 0.17
    slider_field.value = 0.14
    slider_water.value = 0.07
    slider_brush.value = 0.06
    config_perfil["radios"] = [0.3, 1.0, 2.4, 12.8]
    config_perfil["nombre"] = "demencia"
    lbl_radios.value = "Radios cuartiles [25%, 50%, 75%, 95%]: [0.3, 1.0, 2.4, 12.8] km"

def set_perfil_senderista(b):
    slider_road.value = 0.25       # Elementos lineales / Caminos prioritarios
    slider_field.value = 0.14
    slider_structure.value = 0.13
    slider_water.value = 0.08
    slider_woodland.value = 0.07
    slider_brush.value = 0.03
    config_perfil["radios"] = [0.6, 1.8, 3.2, 9.9]
    config_perfil["nombre"] = "senderista"
    lbl_radios.value = "Radios cuartiles [25%, 50%, 75%, 95%]: [0.6, 1.8, 3.2, 9.9] km"

btn_autista.on_click(set_perfil_autista)
btn_demencia.on_click(set_perfil_demencia)
btn_senderista.on_click(set_perfil_senderista)

# Variables globales para almacenar el mapa generado
mapa_actual = {"heatmap": None, "bounds": None, "meter_per_bin": 10, "perfil_nombre": "demo_interactivo"}

def on_generar_mapa_click(b):
    with out_paso2:
        clear_output(wait=True)
        pesos = {
            'structure': slider_structure.value,
            'road': slider_road.value,
            'woodland': slider_woodland.value,
            'water': slider_water.value,
            'field': slider_field.value,
            'brush': slider_brush.value
        }
        total_p = sum(pesos.values())
        if total_p > 0:
            pesos = {k: v/total_p for k, v in pesos.items()}
            
        radios = config_perfil["radios"]
        nombre_p = "demo_interactivo"
        print(f"Generando mapa de calor (Perfil: {config_perfil['nombre'].upper()}, Radios: {radios}) con restricciones fisicas (Agua/Estructuras = 0.0)...")
        utils.generar_mapa_perfil(nombre_p, pesos, radios)
        utils.ejecutar_middleware(nombre_p)
        
        out_dir = f"TFM_JC/resultados/casa_de_campo_{nombre_p}"
        h_path = os.path.join(out_dir, "heatmap.npy")
        g_path = os.path.join(out_dir, "features.geojson")
        
        if os.path.exists(h_path) and os.path.exists(g_path):
            heatmap = np.load(h_path)
            with open(g_path, "r", encoding="utf-8") as f:
                meta = json.load(f)
            bounds = meta["bounds"]
            mapa_actual["heatmap"] = heatmap
            mapa_actual["bounds"] = bounds
            mapa_actual["meter_per_bin"] = meta["meter_per_bin"]
            mapa_actual["perfil_nombre"] = nombre_p
            
            fig, ax = plt.subplots(figsize=(8, 6.5))
            extent = [bounds[0], bounds[2], bounds[1], bounds[3]]
            im = ax.imshow(heatmap, origin="lower", extent=extent, cmap="YlOrRd")
            ax.set_title("Initial Belief Map $b(v^0)$ with Physical Restrictions Filter", fontsize=12, fontweight="bold")
            ax.set_xlabel("UTM Easting (m)", fontsize=10)
            ax.set_ylabel("UTM Northing (m)", fontsize=10)
            ax.grid(True, linestyle=":", alpha=0.4)
            fig.colorbar(im, ax=ax, label="Normalized Probability Density")
            plt.tight_layout()
            display(fig)
            plt.close(fig)
            print("Mapa listo para busqueda y simulacion.")

btn_generar_mapa.on_click(on_generar_mapa_click)

box_botones = widgets.HBox([widgets.Label("Perfiles Rapidos:"), btn_autista, btn_demencia, btn_senderista])
grid_sliders = widgets.VBox([
    widgets.HBox([slider_structure, slider_road]),
    widgets.HBox([slider_woodland, slider_water]),
    widgets.HBox([slider_field, slider_brush]),
    lbl_radios,
    btn_generar_mapa
])

display(widgets.VBox([box_botones, grid_sliders, out_paso2]))

---
## Paso 3: Seleccion, Ejecucion y Visualizacion Dinamica SAR
Configura y lanza cualquier algoritmo de busqueda en tiempo real (continuo de **SAREnv** o bioinspirado de **MTS**). Al ejecutar la simulacion, se renderiza automaticamente:

1. **Panel Izquierdo (Search Trajectory):** Trayectoria completa del dron superpuesta sobre el mapa base $b(v^0)$, marcando el punto de despegue (LKP) y la ubicacion de la victima.
2. **Panel Derecho (Residual Belief Map):** Mapa de creencias actualizado $b(v^{Final})$ tras el barrido dinamico del sensor de 50 metros de radio.
3. **Cuadro de Metricas SAR:** Resumen cuantitativo con el estado de deteccion, longitud del vuelo, area cubierta y porcentaje de probabilidad acumulada.

In [ ]:
# Interfaz de Lanzamiento de Busqueda y Visualizacion en Directo

dropdown_alg = widgets.Dropdown(
    options=[
        ("MTS: Voraz Heuristico (Local Greedy)", "MTS_voraz-heur"),
        ("MTS: Colonia de Hormigas (ACO)", "MTS_ACO"),
        ("MTS: Colonia de Abejas (ABC)", "MTS_ABC"),
        ("MTS: Agujero Negro (BHA)", "MTS_BHA"),
        ("SAREnv: Espiral Continua (Expanding Spiral)", "SAR_spiral"),
        ("SAREnv: Barrido Zigzag (Lawnmower)", "SAR_zigzag"),
        ("SAREnv: Voraz Continuo (Greedy)", "SAR_greedy")
    ],
    value="MTS_voraz-heur",
    description="Algoritmo:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="450px")
)

slider_pasos = widgets.IntSlider(value=250, min=50, max=600, step=25, description="Pasos de Vuelo:")
slider_semilla = widgets.IntSlider(value=0, min=0, max=19, step=1, description="Semilla:")
btn_ejecutar_busqueda = widgets.Button(description="Ejecutar Simulacion", button_style="danger")
out_paso3 = widgets.Output()

def renderizar_resultados_y_metricas():
    heatmap = mapa_actual["heatmap"]
    bounds = mapa_actual["bounds"]
    meter_per_bin = mapa_actual["meter_per_bin"]
    path_coords = mapa_actual["path_coords"]
    goal_utm = mapa_actual["goal_utm"]
    list_x = mapa_actual["list_x"]
    list_y = mapa_actual["list_y"]
    alg_label = mapa_actual["algoritmo_nombre"]
    
    extent = [bounds[0], bounds[2], bounds[1], bounds[3]]
    xs, ys = zip(*path_coords) if path_coords else ([0], [0])
    
    # Calcular mapa de creencias final b(v^end)
    belief_end, visited_mask = vis_pro.compute_updated_belief_map(heatmap, list_x, list_y, len(list_x)-1, sensor_radius_cells=5)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6.5))
    
    # Panel 1: Trayectoria completa sobre el Heatmap Base
    im0 = axes[0].imshow(heatmap, origin="lower", extent=extent, cmap="YlOrRd", alpha=0.85)
    axes[0].plot(xs, ys, color="blue", linewidth=2.2, label="Drone Trajectory", zorder=4)
    axes[0].scatter(xs[0], ys[0], color="blue", marker="o", s=140, edgecolor="white", label="Takeoff (LKP)", zorder=5)
    axes[0].scatter(goal_utm[0], goal_utm[1], color="red", marker="X", s=180, edgecolor="black", label="Victim Position", zorder=5)
    axes[0].scatter(xs[-1], ys[-1], color="orange", marker="^", s=120, edgecolor="black", label="Final Drone Position", zorder=6)
    axes[0].set_title(f"Search Trajectory: {alg_label}", fontsize=12, fontweight="bold")
    axes[0].set_xlabel("UTM Easting (m)", fontsize=10)
    axes[0].set_ylabel("UTM Northing (m)", fontsize=10)
    axes[0].grid(True, linestyle=":", alpha=0.4)
    axes[0].legend(loc="upper right", facecolor="white", framealpha=0.9)
    fig.colorbar(im0, ax=axes[0], label="Initial Density $b(v^0)$")
    
    # Panel 2: Mapa de Creencias Residual b(v^k)
    im1 = axes[1].imshow(belief_end, origin="lower", extent=extent, cmap="YlOrRd", vmin=0, vmax=np.max(heatmap), alpha=0.85)
    if np.any(visited_mask):
        axes[1].imshow(np.ma.masked_where(~visited_mask, visited_mask), origin="lower", extent=extent, cmap="Blues", alpha=0.45, zorder=2)
    axes[1].plot(xs, ys, color="blue", linewidth=1.5, alpha=0.7, zorder=3)
    axes[1].scatter(goal_utm[0], goal_utm[1], color="red", marker="X", s=180, edgecolor="black", label="Victim", zorder=5)
    axes[1].set_title("Residual Belief Map $b(v^{Final})$ (50m Sensor Footprint)", fontsize=12, fontweight="bold")
    axes[1].set_xlabel("UTM Easting (m)", fontsize=10)
    axes[1].grid(True, linestyle=":", alpha=0.4)
    fig.colorbar(im1, ax=axes[1], label="Residual Density")
    
    plt.tight_layout()
    display(fig)
    plt.close(fig)
    
    # Calcular Metricas SAR
    linea = LineString(path_coords) if len(path_coords) >= 2 else LineString()
    punto_v = Point(goal_utm)
    distancia_total_km = linea.length / 1000.0 if not linea.is_empty else 0.0
    dist_min = linea.distance(punto_v) if not linea.is_empty else 99999.0
    encontrada = dist_min <= 50.0
    sensor_area_km2 = (len(np.where(visited_mask)[0]) * (meter_per_bin**2)) / 1e6
    prob_reducida = 1.0 - (np.sum(belief_end) / np.sum(heatmap))
    
    estado_html = "<span style='color:green; font-weight:bold;'>DETECTED (SUCCESS)</span>" if encontrada else "<span style='color:red; font-weight:bold;'>NOT DETECTED</span>"
    
    html_metricas = f"""
    <div style="border: 2px solid #2b5797; border-radius: 8px; padding: 15px; background-color: #f8f9fa; max-width: 700px; margin-top: 15px;">
        <h3 style="color: #2b5797; margin-top: 0;">SAR Evaluation Metrics Dashboard</h3>
        <table style="width: 100%; border-collapse: collapse; font-family: Arial, sans-serif;">
            <tr style="border-bottom: 1px solid #ddd; height: 30px;">
                <td><strong>Detection Status:</strong></td>
                <td>{estado_html}</td>
            </tr>
            <tr style="border-bottom: 1px solid #ddd; height: 30px;">
                <td><strong>Minimum Distance to Victim:</strong></td>
                <td>{dist_min:.1f} meters (Camera FOV: 50 m)</td>
            </tr>
            <tr style="border-bottom: 1px solid #ddd; height: 30px;">
                <td><strong>Flight Path Length:</strong></td>
                <td>{distancia_total_km:.2f} km ({len(path_coords)-1} steps)</td>
            </tr>
            <tr style="border-bottom: 1px solid #ddd; height: 30px;">
                <td><strong>Covered Ground Area:</strong></td>
                <td>{sensor_area_km2:.3f} km²</td>
            </tr>
            <tr style="height: 30px;">
                <td><strong>Accumulated Reduced Probability:</strong></td>
                <td>{prob_reducida*100:.1f}% of belief map</td>
            </tr>
        </table>
    </div>
    """
    display(HTML(html_metricas))

def ejecutar_simulacion_directa(b):
    with out_paso3:
        clear_output(wait=True)
        nombre_p = mapa_actual.get("perfil_nombre", "demo_interactivo")
        json_escenario = f"TFM_JC/resultados/casa_de_campo_{nombre_p}/escenario_{nombre_p}.json"
        
        # Auto-inicializacion transparente si aun no se ha pulsado 'Generar Mapa' en el Paso 2
        if not os.path.exists(json_escenario) or mapa_actual["heatmap"] is None:
            print("Inicializando mapa base de busqueda...")
            pesos_def = {'structure': 0.45, 'road': 0.18, 'water': 0.09, 'brush': 0.09, 'woodland': 0.09, 'field': 0.09}
            radios_def = [0.6, 1.6, 3.7, 15.2]
            utils.generar_mapa_perfil(nombre_p, pesos_def, radios_def)
            utils.ejecutar_middleware(nombre_p)
            
            out_dir = f"TFM_JC/resultados/casa_de_campo_{nombre_p}"
            mapa_actual["heatmap"] = np.load(os.path.join(out_dir, "heatmap.npy"))
            with open(os.path.join(out_dir, "features.geojson"), "r", encoding="utf-8") as f:
                mapa_actual["bounds"] = json.load(f)["bounds"]
            mapa_actual["perfil_nombre"] = nombre_p
            
        heatmap = mapa_actual["heatmap"]
        bounds = mapa_actual["bounds"]
        meter_per_bin = mapa_actual["meter_per_bin"]
        alg_sel = dropdown_alg.value
        pasos = slider_pasos.value
        seed = slider_semilla.value
        minx, miny, maxx, maxy = bounds
        center_x = (minx + maxx) / 2.0
        center_y = (miny + maxy) / 2.0
        
        print(f"-> Ejecutando {dropdown_alg.label} con {pasos} pasos (Semilla: {seed})...")
        
        # Muestrear posicion de la victima en zona transitable
        np.random.seed(seed)
        p_flat = heatmap.flatten() / np.sum(heatmap)
        idx_v = np.random.choice(len(p_flat), p=p_flat)
        r_v, c_v = np.unravel_index(idx_v, heatmap.shape)
        goal_utm = (minx + c_v * meter_per_bin + 0.5 * meter_per_bin, miny + r_v * meter_per_bin + 0.5 * meter_per_bin)
        
        # Ejecucion segun motor (MTS o SAREnv)
        if alg_sel.startswith("MTS_"):
            alg_code = alg_sel.replace("MTS_", "")
            
            with open(json_escenario, "r", encoding="utf-8") as f:
                conf = json.load(f)
            conf["algoritmo_busqueda"] = alg_code
            conf["num_steps"] = pasos
            conf["semilla"] = seed
            conf["dibujar_animacion"] = False
            conf["show_evolution"] = False
            conf["obj_pos"] = [[int(c_v), int(r_v)]]
            
            with open(json_escenario, "w", encoding="utf-8") as f:
                json.dump(conf, f, indent=4)
                
            import subprocess
            res = subprocess.run([sys.executable, "bf-busqueda.py", json_escenario], capture_output=True, text=True)
            if res.returncode != 0:
                print(f"Error al ejecutar MTS: {res.stderr}")
                return
                
            # Localizar el archivo de trayectoria generado
            traj_files = glob.glob(f"TFM_JC/resultados/escenario_{nombre_p}/*-traj.json")
            if not traj_files:
                traj_files = glob.glob(f"TFM_JC/resultados/casa_de_campo_{nombre_p}/*-traj.json")
            
            if not traj_files:
                print("No se encontro el archivo de trayectoria generado.")
                return
                
            traj_file = max(traj_files, key=os.path.getmtime)
            with open(traj_file, "r", encoding="utf-8") as f:
                data_tr = json.load(f)
            list_x = data_tr["list_x"][0]
            list_y = data_tr["list_y"][0]
            path_coords = [(minx + x * meter_per_bin + 0.5 * meter_per_bin, miny + y * meter_per_bin + 0.5 * meter_per_bin) for x, y in zip(list_x, list_y)]
            
        else:
            if alg_sel == "SAR_spiral":
                lines = sap.generate_spiral_path(center_x, center_y, max_radius=2000, fov_deg=90, altitude=50, overlap=0.1, num_drones=1, path_point_spacing_m=10, budget=pasos*10)
            elif alg_sel == "SAR_zigzag":
                lines = sap.generate_pizza_zigzag_path(center_x, center_y, max_radius=2000, num_drones=1, fov_deg=90, altitude=50, overlap=0.1, path_point_spacing_m=10, border_gap_m=10, budget=pasos*10)
            else:
                lines = sap.generate_greedy_path(center_x, center_y, num_drones=1, probability_map=heatmap, bounds=bounds, max_radius=2000, fov_deg=90, altitude=50, budget=pasos*10)
            
            line = lines[0] if isinstance(lines, list) else lines
            path_coords = list(line.coords) if not line.is_empty else [(center_x, center_y)]
            list_x = [int((pt[0] - minx)/meter_per_bin) for pt in path_coords]
            list_y = [int((pt[1] - miny)/meter_per_bin) for pt in path_coords]

        mapa_actual["path_coords"] = path_coords
        mapa_actual["list_x"] = list_x
        mapa_actual["list_y"] = list_y
        mapa_actual["goal_utm"] = goal_utm
        mapa_actual["algoritmo_nombre"] = dropdown_alg.label
        
        print("Simulacion completada con exito. Renderizando visualizacion y metricas...")
        renderizar_resultados_y_metricas()

btn_ejecutar_busqueda.on_click(ejecutar_simulacion_directa)

display(widgets.VBox([
    dropdown_alg,
    widgets.HBox([slider_pasos, slider_semilla]),
    btn_ejecutar_busqueda,
    out_paso3
]))